# MY FARM — Matooke (Banana) Disease Model Training

Trains the matooke classifier on the **Tanzania banana dataset** (Mduma & Leo, NM-AIST) — 16,092 real smartphone photos of banana **leaves and stems**: healthy, **Black Sigatoka** and **Fusarium Wilt Race 1**.

> Stems are included on purpose — Fusarium Wilt often shows on the pseudostem, not just the leaves.

## BEFORE YOU RUN THIS — get the data into Google Drive

1. Dataset page: search **Harvard Dataverse** for *“Bananas Dataset Tanzania”*
   (newer leaves-only alternative: *“Banana Leaves Imagery Dataset”*, 11,767 images — also works)
2. Download the zip files. The original uploads as **6 zips**: 2 healthy, 2 Black Sigatoka, 2 Fusarium Wilt.
3. In **Google Drive**, create a folder: **`myfarm_data/matooke`**
4. Upload the zips there (upload the zips — this notebook unzips them).

Then: **Runtime → Change runtime type → GPU**, and **Runtime → Run all**.

> The paired folders (e.g. healthy_1 / healthy_2) are merged automatically into one class each.

## 1. GPU check

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NO GPU — Runtime > Change runtime type > GPU')

## 2. Download the dataset straight into Colab
No Google Drive, no manual download — Colab pulls it from Harvard Dataverse using the dataset's DOI.
This is a large download; let it run.

In [ ]:
# Download the Bananas Dataset Tanzania DIRECTLY into Colab from Harvard Dataverse.
# DOI: 10.7910/DVN/LQUWXW  (NM-AIST Bananas dataset)
# Colab's connection does the work — your own internet is not used for the GBs.
import os, shutil

DOI = 'doi:10.7910/DVN/LQUWXW'
RAW = '/content/matooke_raw'
shutil.rmtree(RAW, ignore_errors=True)
os.makedirs(RAW, exist_ok=True)

# Dataverse API: download the whole dataset as one zip
url = f'https://dataverse.harvard.edu/api/access/dataset/:persistentId/?persistentId={DOI}'
print('Downloading from Harvard Dataverse (this is large — give it time)...')
!wget -q --show-progress -O /content/matooke_all.zip "{url}"

print('\nDownloaded. Size:')
!ls -lh /content/matooke_all.zip

# Unzip the outer archive
import zipfile
with zipfile.ZipFile('/content/matooke_all.zip') as zf:
    zf.extractall(RAW)
print('Outer zip extracted.')

# The dataset ships as several inner zips — extract those too
import glob
for z in glob.glob(f'{RAW}/**/*.zip', recursive=True):
    print('Unzipping inner:', os.path.basename(z))
    try:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(os.path.dirname(z))
    except Exception as e:
        print('  skipped:', e)
print('All extracted.')

## 3. Sort images into the 3 classes
Merges the paired folders (e.g. healthy_1 + healthy_2) into one class each.

In [ ]:
# Sort every image into 3 class folders, merging the paired uploads.
DATA = '/content/matooke_data'
shutil.rmtree(DATA, ignore_errors=True)

def target_class(name):
    n = name.upper().replace('-', '_').replace(' ', '_')
    if 'SIGATOKA' in n: return 'BLACK_SIGATOKA'
    if 'FUSARIUM' in n or 'WILT' in n: return 'FUSARIUM_WILT'
    if 'HEALTH' in n: return 'HEALTHY'
    return None

count = {'HEALTHY':0, 'BLACK_SIGATOKA':0, 'FUSARIUM_WILT':0}
for root, dirs, files in os.walk(RAW):
    cls = None
    for part in root.replace('\\','/').split('/'):
        t = target_class(part)
        if t: cls = t
    for f in files:
        if not f.lower().endswith(('.jpg','.jpeg','.png')):
            continue
        c = cls or target_class(f)   # fall back to the filename
        if not c: continue
        d = f'{DATA}/{c}'
        os.makedirs(d, exist_ok=True)
        shutil.copy(os.path.join(root, f), f'{d}/{c}_{count[c]}.jpg')
        count[c] += 1

print('Images per class:', count)
assert sum(count.values()) > 0, 'No images found — check the download/extract steps above.'

## 4. Build the data pipeline
85/15 split. Keras sorts folders alphabetically: BLACK_SIGATOKA, FUSARIUM_WILT, HEALTHY — mapped to the app's treatment keys.

In [ ]:
import numpy as np
IMG_SIZE = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA, validation_split=0.15, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA, validation_split=0.15, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH)

folder_names = train_ds.class_names
print('Folder order:', folder_names)

# Must match keys in the app's treatment_db.dart
APP_KEY = {
    'BLACK_SIGATOKA': 'banana_black_sigatoka',
    'FUSARIUM_WILT': 'banana_fusarium_wilt',
    'HEALTHY': 'healthy',
}
class_names = [APP_KEY[f] for f in folder_names]
print('Label keys (match treatment_db.dart):', class_names)

from sklearn.utils.class_weight import compute_class_weight
y = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
w = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y)
class_weight = {i: float(x) for i, x in enumerate(w)}
print('Class weights:', class_weight)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 5. Build the model (transfer learning)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

data_aug = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augment')

base = EfficientNetB0(include_top=False, weights='imagenet',
                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = data_aug(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base(x)   # do NOT pass training=False
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Phase 1 — train the head

In [ ]:
cbs1 = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, min_lr=1e-5)]
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=cbs1)

## 7. Phase 2 — fine-tune

In [ ]:
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cbs2 = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, min_lr=1e-6)]
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=cbs2)

## 8. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
y_true, y_pred = [], []
for imgs, labels in val_ds:
    p = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(p, axis=1)); y_true.extend(labels.numpy())
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,4)); ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right'); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix — Matooke')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j,i,cm[i,j],ha='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout(); plt.savefig('matooke_confusion_matrix.png', dpi=120); plt.show()

## 9. Export to TensorFlow Lite

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = conv.convert()
open('matooke.tflite','wb').write(tflite)
open('matooke_labels.txt','w').write('\n'.join(class_names))
print('matooke.tflite', round(len(tflite)/1e6,1), 'MB'); print('Labels:', class_names)

## 10. Download for the app

In [ ]:
from google.colab import files
files.download('matooke.tflite')
files.download('matooke_labels.txt')
files.download('matooke_confusion_matrix.png')
print('Put matooke.tflite and matooke_labels.txt into: myfarm_flutter/assets/models/')